# DPO orca-math-korean

**Dataset:**
- https://huggingface.co/datasets/microsoft/orca-math-word-problems-200k
- https://huggingface.co/datasets/kuotient/orca-math-korean-dpo-pairs

**Model:**
- https://huggingface.co/soka0000/vclm-korean-7b

In [ ]:
%pip install -Uqq datasets transformers hf_transfer accelerate peft trl wandb scikit-learn

### 데이터 준비

In [2]:
from datasets import load_dataset # HuggingFace 데이터셋 로드 함수

# DPO용 학습 데이터 로드
dataset = load_dataset('kuotient/orca-math-korean-dpo-pairs', split='train')
SAMPLE_SIZE = 10000
dataset = dataset.select(range(SAMPLE_SIZE))
print(len(dataset))
print(dataset[100])


10000
{'system': '당신은 인공지능 어시스턴트입니다.', 'question': '한 배럴에는 12리터(L)와 400밀리리터(ml)의 석유, B 배럴에는 7600밀리리터(ml)의 석유가 들어 있습니다. A 배럴과 B 배럴의 석유 양을 같게 하려면 A 배럴에서 B 배럴로 몇 리터(L)를 옮겨야 합니까?', 'chosen': '먼저 모든 측정값을 동일한 단위로 변환하여 계산을 쉽게 해봅시다. 모든 것을 밀리리터(ml)로 변환하겠습니다.\n\n배럴에는 12리터와 400밀리리터가 들어 있습니다. 1리터는 1000밀리리터와 같으므로 12리터를 밀리리터로 변환할 수 있습니다:\n12리터 = 12 * 1000밀리리터 = 12000밀리리터\n\n이제 이미 밀리리터 단위로 표시된 400밀리리터를 더합니다:\n12000밀리리터 + 400밀리리터 = 12400밀리리터\n\n따라서 배럴에는 총 12400밀리리터의 석유 가 들어 있습니다.\n\nB 배럴에는 7600밀리리터의 석유가 들어 있습니다.\n\n두 배럴의 석유 양을 동일하게 하려면 두 양의 평균을 구해야 합니다:\n두 배럴의 총량 = 12400밀리리터(A배럴) + 7600밀리리터(B배럴)\n두 배럴의 총량 = 20000밀리리터\n\n이제 이 총량을 2로 나누어 각 배럴에 해당하는 양을 구합니다:\n각 배럴의 동일한 양 = 20000밀리리터 / 2\n각 배럴의 동일한 양 = 10000밀리리터\n\n현재 A 배럴에는 12400밀리리터가 있으므로, 두 배럴의 양이 각각 10000밀리리터가 되도록 일부를 B 배럴로 이동해야 합니다.\n\nA에서 B로 이동할 양 = 12400밀리리터(A 배럴) - 10000밀리리터(동일한 양)\nA에서 B로 이동할 양 = 2400밀리리터\n\n따라서 두 배럴의 석유 양을 동일하게 하기 위해 A 배럴에서 B 배럴로 2400밀리리터(또는 2.4리터)를 이동해야 합니다.', 'rejected': ' 먼저 B 배럴의 석유 양을 리터로 변환하여 A 배럴의 석유 양과 비교해야 합니다.\n\nB 배럴의

## 모델 준비

https://huggingface.co/soka0000/vclm-korean-7b

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'soka0000/vclm-korean-7b'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16,  # 메모리 절약 및 연산 효율성
    device_map = 'auto',     # CPU / GPU 자동 배치
    trust_remote_code = True # 모델 저장소의 커스텀 코드 허용
)

[transformers] You are using a model of type `Soka1.0` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


## 데이터 전처리
- Chat Template 적용
- prompt /chosen / rejected 형태로 변환

In [5]:
def preprocess_dpo_data(example):
    messages = [
        {"role": 'system', 'content': example['system']},
        {"role": 'user', 'content': example['question']},
    ]
    prompt_style = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return {
        'prompt': prompt_style,
        'chosen': example['chosen'],
        'rejected': example['rejected']
    }

dataset_preprocessed = dataset.map(preprocess_dpo_data)

print(f"변경 전 컬럼 : {dataset.column_names}")
print(f"변경 후 컬럼 : {dataset_preprocessed.column_names}")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

변경 전 컬럼 : ['system', 'question', 'chosen', 'rejected']
변경 후 컬럼 : ['system', 'question', 'chosen', 'rejected', 'prompt']


In [6]:
dataset_preprocessed['prompt'][100]

'<|im_start|>system\n당신은 인공지능 어시스턴트입니다.<|im_end|>\n<|im_start|>user\n한 배럴에는 12리터(L)와 400밀리리터(ml)의 석유, B 배럴에는 7600밀리리터(ml)의 석유가 들어 있습니다. A 배럴과 B 배럴의 석유 양을 같게 하려면 A 배럴에서 B 배럴로 몇 리터(L)를 옮겨야 합니까?<|im_end|>\n<|im_start|>assistant\n'

In [8]:
# 전처리된 데이터셋을 train/valid/test 데이터로 분할 (8:1:1)
train_size = int(len(dataset_preprocessed)*0.8)
val_size = int(len(dataset_preprocessed)*0.1)
test_size = int(len(dataset_preprocessed)*0.1)

train_dataset = dataset_preprocessed.select(range(train_size))
val_dataset = dataset_preprocessed.select(range(train_size, train_size + val_size))
test_dataset = dataset_preprocessed.select(range(train_size + val_size, len(dataset_preprocessed)))

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

8000
1000
1000


### BaseModel 학습 전 테스트

In [9]:
def generate_response(model, tokenizer, question):
    prompt = question
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, # input_ids, attention_mask, ...
            max_length = 1024, # 생성 토큰 포함 전체 토큰 길이
            do_sample = True,  # 창의적 생성
            top_k = 50,        # 상위 50개 토큰 중 선택
            top_p = 0.95,      # 누적확률 95% 이상인 토큰 중 선택
            temperature = 0.5, # 약간 창의적
            num_return_sequences = 1, # 답변 1개
            eos_token_id = tokenizer.eos_token_id, # 종료 토큰
            pad_token_id = tokenizer.pad_token_id, # 패딩 토큰
        )
        generated_text = tokenizer.decode(outputs[0]) # 생성된 토큰을 문자열로 디코딩
        return generated_text.replace(prompt, '').strip()

In [ ]:
# 모델 학습 전 샘플 3개 테스트
test_subset = test_dataset.select(range(3))

for i, example in enumerate(test_subset):
    question = example['prompt']
    answer = example['chosen']

    print(f"질문 : {question} / 정답 : {answer}")

    generate_answer = generate_response(model, tokenizer, question)
    print(f"모델 생성 답변 : {generate_answer}")
    print("="*30)

## DPO 학습 준비

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r = 8,          # 추가 학습할 저차원 행렬 크기
    lora_alpha= 16,
    target_modules= ['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias = 'none', # bias 학습하지 않음
    task_type= 'CAUSAL_LM' # 작업 유형: 생성형 LM
)
model = get_peft_model(model, lora_config) # 기존 모델에 LoRA 적용

model.print_trainable_parameters() # 학습가능 파라미터 수/비율 확인

In [ ]:
# Reference(참조) 모델 로드 : 정책모델이 얼마나 바뀌엇는지 비교 기준이 되는 고정 모델
# chosen/rejected 답변에 대한 생성확률 비교값 제공

reference_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16,  # 메모리 절약 및 연산 효율성
    device_map = 'auto',     # CPU / GPU 자동 배치
    trust_remote_code = True # 모델 저장소의 커스텀 코드 허용
)
# 학습 방지 처리
reference_model.eval() # 평가 모드로 전환
for param in reference_model.parameters():
    param.requires_grad = False

In [ ]:
import wandb

wandb.login()

In [ ]:
# DPOConfig 설정 / DPOTrainer 학습 실행
from trl import DPOTrainer, DPOConfig

hub_model_id = 'lllee2/vclm-korean-7b-orca-math-korean-dpo'

training_args = DPOConfig(  # DPO 학습 하이퍼파라미터/로깅/저장 설정
    output_dir='vclm-korean-7b-orca-math-korean-dpo',  # 체크포인트/로그 저장 폴더
    num_train_epochs=1,  # 전체 데이터를 1번 반복 학습
    per_device_train_batch_size=2,  # GPU 1개당 배치 크기
    gradient_accumulation_steps=4,  # 4번 누적 후 업데이트(실제 배치 효과: 2*4=8)
    learning_rate=5e-5,  # 학습률
    eval_strategy="steps",  # 일정 step마다 평가 수행
    save_strategy="steps",  # 일정 step마다 저장 수행
    logging_steps=50,  # 50 step마다 학습 로그 출력
    fp16=False,  # fp16 비활성화(여기서는 bf16 사용)
    bf16=True,  # bfloat16 사용(지원 GPU에서 안정적/빠름)
    tf32=True,  # Ampere 이상에서 matmul 가속 옵션(정밀도 약간 완화)
    beta=0.1,  # DPO의 beta(선호 강도 조절)
    max_length=512,  # prompt+답변을 포함한 최대 길이
    max_prompt_length=256,  # prompt에 할당할 최대 길이
    remove_unused_columns=False,  # DPO에 필요한 컬럼이 제거되지 않도록 유지
    push_to_hub=True,  # 학습 결과를 Hugging Face Hub로 업로드
    hub_model_id=hub_model_id,  # 업로드할 저장소 이름
    hub_strategy="end",  # 학습 끝난 뒤 한 번만 업로드
    report_to=['wandb']  # wandb로 학습 로그 전송
)

dpo_trainer = DPOTrainer(  # DPO Trainer 생성(정책모델 vs 참조모델 비교 학습)
    model=model,  # 정책모델(LoRA 적용된 학습 대상)
    ref_model=reference_model,  # 참조모델(고정, 비교 기준)
    args=training_args,  # 위에서 만든 학습 설정
    train_dataset=train_dataset,  # 학습 데이터(prompt/chosen/rejected)
    eval_dataset=val_dataset,  # 검증 데이터
    processing_class=tokenizer  # 토큰화 처리(TRL 버전에 따라 tokenizer 인자명 상이 가능)
)

dpo_trainer.train()  # DPO 학습 시작

### DPO 학습 후 Model 테스트

In [ ]:
# 모델 학습 전 샘플 3개 테스트
test_subset = test_dataset.select(range(3))

for i, example in enumerate(test_subset):
    question = example['prompt']
    answer = example['chosen']

    print(f"질문 : {question} / 정답 : {answer}")

    generate_answer = generate_response(model, tokenizer, question)
    print(f"모델 생성 답변 : {generate_answer}")
    print("="*30)

### DPO 성능 평가

In [ ]:
 chosen vs rejected 선호 정확도 평가
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

def calculate_log_prob(model, tokenizer, prompt, response):
    """
    주어진 prompt에 대한 response의 log probability(평균)값을 계산합니다.

    Args:
        model: HuggingFace AutoModelForCausalLM (or similar)
        tokenizer: HuggingFace AutoTokenizer
        prompt (str): 프롬프트 텍스트
        response (str): 응답 텍스트
    
    Returns:
        float: response 토큰들의 평균 log probability
    """
    full_text = prompt + " " + response
    inputs = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=512).to(model.device)  # 토큰화 후 장치 이동
    input_ids = inputs['input_ids']  # 토큰 ID들
    attention_mask = inputs['attention_mask']  # 패딩 무시용 마스크

    prompt_tokens = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)  # 토큰화 후 장치 이동
    prompt_len = prompt_tokens['input_ids'].shape[1]  # 프롬프트 토큰 개수 확인 (response와의 경계)

    # 학습 안하고, 계산만 진행
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # 순전파
        logits = outputs.logits    # 각 위치별 다음 토큰 후보들 점수들

    # contiguous() : 텐서 데이터를 메모리상 연속된 형태로 재정리 함수
    shift_logits = logits[..., :-1, :].contiguous()  # 마지막 위치 제외(다음 토큰 예측용 길이 맞춰줌)
    shift_labels = input_ids[..., 1:].contiguous()   # 정답 토큰을 한 칸 왼쪽으로 당김 (다음토큰)

    log_probs = F.log_softmax(shift_logits, dim=-1)  # 점수표 -> log확률표 (확률로 바꾼 뒤 log)

    # 각 위치별 정답 토큰 log 확률만 뽑아냄
    true_log_probs = torch.gather(
        log_probs,  # (batch, seq_len-1, vocab)
        2,          # vocab 차원에서 선택
        shift_labels.unsqueeze(-1)  # (batch, seq_len -1) -> (batch, seq_len -1, 1)
    ).squeeze(-1)   # (batch, seq_len -1, 1) -> (batch, seq_len -1)

    seq_len = shift_labels.shape[1]  # 실제 점수 길이 (seq_len - 1)
    # 프롬프트가 너무 길어서 response 구간이 없는 경우
    if prompt_len >= seq_len:
        valid_log_probs = true_log_probs[:, -1:]  # 마지막 값만 사용
    else:
        start_idx = max(0, prompt_len - 1)  # response 첫 토큰 점수 위치 (프롬프트의 마지막 토큰 자리)
        valid_log_probs = true_log_probs[:, start_idx:]  # prompt 점수 제외 response 점수만 사용

    avg_log_prob = valid_log_probs.mean().item()  # response 점수 평균

    return avg_log_prob  # 평균 log-prob

In [ ]:
def calculate_preferance_accuracy(model, tokenizer, dataset, num_samples=100):
    """
    모델이 선호(chosen) 답변에 비선호(rejected) 답변보다 더 높은 확률을 부여하는지 평가
    """
    correct = 0
    total = min(num_samples, len(dataset))

    print(f"전체 개수 : {total}")

    model.eval()

    for idx in range(total):
        example = dataset[idx]
        prompt = example['prompt']
        chosen = example['chosen']
        rejected = example['rejected']

        chosen_score = calculate_log_prob(model, tokenizer, prompt, chosen)
        rejected_score = calculate_log_prob(model, tokenizer, prompt, rejected)

        if chosen_score > rejected_score:
            correct += 1

        if (idx + 1) % 10 == 0:
            print(f"{idx+1}/{total} 정확도 : {correct/(idx+1)*100:.2f}")
    accuracy = correct / total
    return accuracy

test_accuracy = calculate_preferance_accuracy(model, tokenizer, test_dataset, num_samples=300)
print(test_accuracy)